# DocTamper — Random Image Forgery Checker

This notebook:
1. Installs dependencies
2. Downloads the DocTamper dataset via KaggleHub
3. Loads your saved model `doctamper_tampernet_overall_best.pth` (upload it to Colab files first)
4. Picks **5 random images** from the TestingSet each run
5. For each image shows: Original | Ground-Truth Tamper Region | Predicted Tamper Region + verdict

> **Re-run the last cell** any time to get a fresh random batch!

## Step 1 — Install dependencies

In [ ]:
!pip -q install kagglehub lmdb albumentations segmentation-models-pytorch==0.3.3 timm opencv-python-headless scikit-learn tqdm

## Step 2 — Imports & device setup

In [ ]:
import os
import random
import time
from pathlib import Path
from io import BytesIO

import cv2
import lmdb
import kagglehub
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

# ---- Device ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: Running on CPU — inference will be slow. Enable GPU in Runtime > Change runtime type.")

## Step 3 — Download DocTamper dataset

In [ ]:
print("Downloading DocTamper dataset via KaggleHub (this may take a few minutes)...")
dataset_path = kagglehub.dataset_download("dinmkeljiame/doctamper")
data_root = Path(dataset_path)
print(f"\nDataset ready at: {data_root}")
print("Contents:", [p.name for p in sorted(data_root.iterdir())])

## Step 4 — Open LMDB environments

In [ ]:
def find_lmdb_dirs(root: Path):
    found = {}
    for p in root.rglob("*"):
        if p.is_dir() and (p / "data.mdb").exists() and (p / "lock.mdb").exists():
            found[p.name] = p
    return found

def open_env(folder: Path):
    return lmdb.open(
        str(folder),
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False,
    )

lmdb_dirs = find_lmdb_dirs(data_root)
environments = {name: open_env(path) for name, path in lmdb_dirs.items()}
print("Opened LMDB environments:", list(environments.keys()))

## Step 5 — Model definition (must match training architecture)

In [ ]:
class TamperNet(nn.Module):
    """Exact same architecture used during training."""
    def __init__(self, encoder_name="efficientnet-b0", encoder_weights=None):
        super().__init__()
        self.seg = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,  # None at inference — weights come from .pth
            in_channels=3,
            classes=1,
        )
        enc_ch = self.seg.encoder.out_channels[-1]
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(enc_ch, 1),
        )

    def forward(self, x):
        features = self.seg.encoder(x)
        dec_out = self.seg.decoder(*features)
        mask_logits = self.seg.segmentation_head(dec_out)
        cls_logits = self.cls_head(features[-1])
        return mask_logits, cls_logits

print("TamperNet class defined.")

## Step 6 — Load saved model weights

Make sure `doctamper_tampernet_overall_best.pth` is uploaded to your Colab session files (left sidebar → Files → Upload).

In [ ]:
MODEL_PATH = "/content/doctamper_tampernet_overall_best.pth"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}.\n"
        "Please upload 'doctamper_tampernet_overall_best.pth' to Colab files first."
    )

model = TamperNet(encoder_name="efficientnet-b0", encoder_weights=None).to(DEVICE)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

# Handle both raw state_dict and wrapped checkpoint dicts
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint  # assume it's a raw state_dict

model.load_state_dict(state_dict)
model.eval()
print(f"Model loaded successfully from: {MODEL_PATH}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 7 — Helper utilities

In [ ]:
# ---------- LMDB helpers ----------

def get_num_samples(env: lmdb.Environment) -> int:
    with env.begin(write=False) as txn:
        count = 0
        with txn.cursor() as cur:
            for k, _ in cur:
                if k.startswith(b"image-"):
                    count += 1
        return count

def read_image_label(env: lmdb.Environment, index: int):
    key_img = f"image-{index:09d}".encode("utf-8")
    key_lbl = f"label-{index:09d}".encode("utf-8")
    with env.begin(write=False) as txn:
        image_buffer = txn.get(key_img)
        label_buffer = txn.get(key_lbl)
    if image_buffer is None:
        raise KeyError(f"No image for index={index}")
    image = Image.open(BytesIO(image_buffer)).convert("RGB")
    image_np = np.array(image)
    mask_np = None
    if label_buffer is not None:
        label = Image.open(BytesIO(label_buffer)).convert("L")
        mask_np = (np.array(label) > 127).astype(np.uint8)
    return image_np, mask_np

# ---------- Preprocessing ----------

IMG_SIZE = 512  # same as training

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

def preprocess(image_np: np.ndarray) -> torch.Tensor:
    """Resize + normalise + add batch dim."""
    augmented = val_transform(image=image_np)
    return augmented["image"].unsqueeze(0).to(DEVICE)

# ---------- Visualisation helpers ----------

def overlay_mask(image_np: np.ndarray, mask_np: np.ndarray,
                 color=(220, 30, 30), alpha: float = 0.45):
    """Blend a binary mask onto the image with a solid colour."""
    overlay = image_np.copy().astype(np.float32)
    c = np.array(color, dtype=np.float32)
    for ch in range(3):
        overlay[..., ch] = np.where(
            mask_np > 0,
            (1 - alpha) * overlay[..., ch] + alpha * c[ch],
            overlay[..., ch]
        )
    return overlay.clip(0, 255).astype(np.uint8)

def add_border(image_np: np.ndarray, color: tuple, thickness: int = 8):
    """Add a coloured border to highlight verdict."""
    img = image_np.copy()
    img[:thickness, :] = color
    img[-thickness:, :] = color
    img[:, :thickness] = color
    img[:, -thickness:] = color
    return img

print("Helpers defined.")

## Step 8 — Run inference on 5 random images

Re-run this cell as many times as you like — each run picks a different random set.

In [ ]:
# ============================================================
#  CONFIG — change these if you like
# ============================================================
N_IMAGES       = 5           # how many images to sample per run
PRED_THRESHOLD = 0.5         # pixel-level threshold for "tampered" pixel
CLS_THRESHOLD  = 0.5         # image-level threshold for "tampered" verdict

# Which LMDB split to sample from
# Options: "DocTamperV1-TestingSet", "DocTamperV1-SCD",
#          "DocTamperV1-FCD",  "DocTamperV1-TrainingSet"
SPLIT_NAME = "DocTamperV1-TestingSet"
# ============================================================

# --- Resolve which environment to use ---
if SPLIT_NAME in environments:
    test_env = environments[SPLIT_NAME]
else:
    # Fall back to whatever is available
    SPLIT_NAME = list(environments.keys())[0]
    test_env   = environments[SPLIT_NAME]
    print(f"Requested split not found. Using: {SPLIT_NAME}")

# Seed with current time so every run is different
rng_seed = int(time.time())
random.seed(rng_seed)
np.random.seed(rng_seed)
print(f"Random seed this run: {rng_seed}")

# --- Count samples & pick random indices ---
n_total = get_num_samples(test_env)
print(f"Split '{SPLIT_NAME}' has {n_total:,} images.")
sampled_indices = random.sample(range(1, n_total + 1), min(N_IMAGES, n_total))
print(f"Sampled indices: {sampled_indices}\n")

# --- Inference loop ---
results = []

with torch.no_grad():
    for idx in sampled_indices:
        image_np, gt_mask = read_image_label(test_env, idx)

        tensor = preprocess(image_np)

        mask_logits, cls_logits = model(tensor)

        # Classification
        cls_prob   = torch.sigmoid(cls_logits).item()
        is_forged  = cls_prob >= CLS_THRESHOLD

        # Segmentation mask — upsample back to original size
        h, w       = image_np.shape[:2]
        pred_mask_full = F.interpolate(
            torch.sigmoid(mask_logits),
            size=(h, w),
            mode="bilinear",
            align_corners=False,
        )[0, 0].cpu().numpy()
        pred_mask_bin = (pred_mask_full >= PRED_THRESHOLD).astype(np.uint8)

        tampered_pct = pred_mask_bin.mean() * 100

        results.append({
            "idx":           idx,
            "image_np":      image_np,
            "gt_mask":       gt_mask,
            "pred_mask_bin": pred_mask_bin,
            "pred_mask_raw": pred_mask_full,
            "cls_prob":      cls_prob,
            "is_forged":     is_forged,
            "tampered_pct":  tampered_pct,
        })

print(f"Inference done on {len(results)} images.")

# --- Plot results ---
COLS        = 4   # Original | GT Region | Pred Region | Heatmap
fig, axes   = plt.subplots(N_IMAGES, COLS, figsize=(COLS * 4.5, N_IMAGES * 4.2))
if N_IMAGES == 1:
    axes = axes[np.newaxis, :]

col_titles = ["Original Image", "GT Tamper Region", "Predicted Region", "Prob Heatmap"]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=13, fontweight="bold", pad=8)

for row, r in enumerate(results):
    img         = r["image_np"]
    gt          = r["gt_mask"]
    pred_bin    = r["pred_mask_bin"]
    pred_raw    = r["pred_mask_raw"]
    verdict     = "🔴 FORGED" if r["is_forged"] else "🟢 AUTHENTIC"
    border_col  = (200, 30, 30) if r["is_forged"] else (30, 160, 30)

    # --- Col 0: Original with verdict border ---
    img_bordered = add_border(img, border_col)
    axes[row, 0].imshow(img_bordered)
    axes[row, 0].set_ylabel(
        f"idx={r['idx']}",
        fontsize=9, rotation=0, labelpad=45, va="center"
    )
    axes[row, 0].set_xlabel(
        f"{verdict}  |  confidence: {r['cls_prob']:.1%}",
        fontsize=10, color=("darkred" if r["is_forged"] else "darkgreen"),
        fontweight="bold"
    )

    # --- Col 1: Ground-truth overlay ---
    if gt is not None:
        gt_has_tamper = gt.max() > 0
        gt_vis = overlay_mask(img, gt, color=(220, 40, 40))
        axes[row, 1].imshow(gt_vis)
        gt_label = f"GT: {'Tampered' if gt_has_tamper else 'Authentic'}"
        axes[row, 1].set_xlabel(gt_label, fontsize=9, color="black")
    else:
        axes[row, 1].imshow(img)
        axes[row, 1].set_xlabel("No GT available", fontsize=9)

    # --- Col 2: Predicted binary mask overlay ---
    pred_vis = overlay_mask(img, pred_bin, color=(220, 40, 40))
    axes[row, 2].imshow(pred_vis)
    axes[row, 2].set_xlabel(
        f"Tampered pixels: {r['tampered_pct']:.1f}%",
        fontsize=9, color="black"
    )

    # --- Col 3: Probability heatmap ---
    im = axes[row, 3].imshow(pred_raw, cmap="hot", vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[row, 3], fraction=0.046, pad=0.04)
    axes[row, 3].set_xlabel("Pixel tamper probability", fontsize=9)

    for col in range(COLS):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

plt.suptitle(
    f"DocTamper — Random Inference  |  Split: {SPLIT_NAME}  |  Seed: {rng_seed}",
    fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

# --- Summary table ---
print("\n" + "="*65)
print(f"{'Idx':>6}  {'Verdict':<16}  {'Cls Conf':>9}  {'Tampered Px':>12}")
print("-"*65)
for r in results:
    verdict_str = "FORGED" if r["is_forged"] else "AUTHENTIC"
    print(f"{r['idx']:>6}  {verdict_str:<16}  {r['cls_prob']:>8.1%}  {r['tampered_pct']:>11.1f}%")
print("="*65)

n_forged = sum(1 for r in results if r["is_forged"])
print(f"\nSummary: {n_forged}/{len(results)} images flagged as FORGED in this batch.")